In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [2]:
columns = [
    'sepal_length',
    'sepal_width',
    'petal_length',
    'petal_width',
    'species'
]

df = pd.read_csv('iris/iris.data', header=None, names=columns)

X = df[['petal_length', 'petal_width']]
y = df['species']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [3]:
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=200))
    ]),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=4,
        random_state=42
    ),

    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier(n_neighbors=5))
    ])
}

In [4]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

results = []

for name, model in models.items():

    cv_scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy'
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_pred)

    results.append({
        'Model': name,
        'CV Mean Accuracy': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Accuracy': test_accuracy
    })

In [5]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='CV Mean Accuracy',
    ascending=False
)

print(results_df.round(4).to_string(index=False))

              Model  CV Mean Accuracy  CV Std  Test Accuracy
                KNN            0.9667  0.0167         0.9667
Logistic Regression            0.9583  0.0264         0.9333
      Decision Tree            0.9250  0.0312         0.9333


In [6]:
best_model_name = results_df.iloc[0]['Model']

print("Best Model based on CV Mean Accuracy:", best_model_name)

Best Model based on CV Mean Accuracy: KNN


In [7]:
best_model = models['KNN']

print(best_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', KNeighborsClassifier())])


In [8]:
import os
import joblib

os.makedirs('models', exist_ok=True)

joblib.dump(best_model, 'models/best_knn_model.pkl')

print("KNN model saved successfully!")

KNN model saved successfully!


In [9]:
import joblib

loaded_model = joblib.load('models/best_knn_model.pkl')

print("Model loaded successfully!")


Model loaded successfully!


In [10]:
new_flower = pd.DataFrame({
    'petal_length': [1.5],
    'petal_width': [0.2]
})

prediction = loaded_model.predict(new_flower)

print("Predicted Species:", prediction[0])

Predicted Species: Iris-setosa
